# GraphRAG — Part 2: Retrieval (Querying)

This notebook queries the knowledge graph built in **`01_graphrag_ingestion.ipynb`**, using GraphRAG's Python
`api` module directly (the same functions the `graphrag query` CLI calls under the hood).

GraphRAG supports four search strategies:

| Method | Best for | Data it uses |
|---|---|---|
| **Global Search** | Holistic, corpus-wide questions ("what are the main themes?") | community reports |
| **Local Search** | Specific, entity-centered questions ("who is X, and how do they relate to Y?") | entities, relationships, text units, community reports |
| **Basic Search** | A plain vector-similarity baseline, for comparison | text units only |
| **DRIFT Search** | Entity-focused questions that benefit from community-level context; the most thorough (and slowest/most expensive) method | all of the above |

This notebook covers **Global** and **Local** as the essentials, with **Basic** and **DRIFT** included at the
end as a bonus for further exploration.

### Prerequisite
You need a completed run of the ingestion notebook — specifically, `<PROJECT_ROOT>/output/*.parquet` must exist.


In [2]:
from pathlib import Path

LOCAL_BASE_DIR = Path(".")
PROJECT_ROOT = LOCAL_BASE_DIR / "graphrag_project"   # always read from local disk

assert (PROJECT_ROOT / "output").exists(), (
    f"No output/ folder found under {PROJECT_ROOT.resolve()} — run the ingestion notebook first "
    f"(and make sure its Step 10 backed up to Google Drive, if you're in a new session)."
)

print(f"Project root       : {PROJECT_ROOT.resolve()}")


Project root       : C:\Users\hi\Desktop\projects\python_projects\tutorial\play_langchain_llamaindex_langgraph\graphrag_project


In [5]:
# Do  not need to run this on local because it is already installed
# %pip install -q -U graphrag pyyaml

# IMPORTANT: After this, restart the session . Otherwise other cells will give error.

In [2]:
import importlib.metadata

# List the distribution package names
packages = ["graphrag", "pyyaml", "langchain-community"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


graphrag version: 3.1.1
pyyaml version: 6.0.3
langchain-community version: 0.4.2


## Step 2 — Load the config and the indexed tables

`load_config` reads `settings.yaml`/`.env` the same way the CLI does. `DataReader` then reads the Parquet
output as pandas DataFrames — these DataFrames are exactly what you'd get from the `_resolve_output_files`
helper inside GraphRAG's own CLI, so this is the same data path the `graphrag query` command uses.


In [3]:
import asyncio

import graphrag.api as api
from graphrag.config.load_config import load_config
from graphrag.data_model.data_reader import DataReader
from graphrag_storage import create_storage
from graphrag_storage.tables.table_provider_factory import create_table_provider

config = load_config(root_dir=PROJECT_ROOT)

storage_obj = create_storage(config.output_storage)
table_provider = create_table_provider(config.table_provider, storage=storage_obj)
reader = DataReader(table_provider)

entities = asyncio.run(reader.entities())
communities = asyncio.run(reader.communities())
community_reports = asyncio.run(reader.community_reports())
text_units = asyncio.run(reader.text_units())
relationships = asyncio.run(reader.relationships())

print(f"entities={len(entities)}  relationships={len(relationships)}  "
      f"communities={len(communities)}  community_reports={len(community_reports)}  "
      f"text_units={len(text_units)}")


entities=10  relationships=10  communities=3  community_reports=3  text_units=1


In [4]:
# A quick look at what's actually in the graph before querying it
entities[["title", "type", "description"]].head(15)


,title,type,description
0,MERIDIAN HEALTH SYSTEMS,ORGANIZATION,Meridian Health Systems is a hospital network ...
1,NOVACARE ROBOTICS,ORGANIZATION,NovaCare Robotics is a company founded in 2018...
2,DR. AMARA OSEI,PERSON,Dr. Amara Osei is the Chief Innovation Officer...
3,WEI ZHANG,PERSON,Wei Zhang is the engineer who founded NovaCare...
4,TEXAS NURSES COALITION,ORGANIZATION,The Texas Nurses Coalition is a group that rai...
5,CEDAR RIDGE MEDICAL,ORGANIZATION,Cedar Ridge Medical is a hospital that signed ...
6,NATIONAL HEALTHCARE INNOVATION SUMMIT,EVENT,The National Healthcare Innovation Summit is a...
7,AUSTIN,GEO,Austin is the city in Texas where Meridian Hea...
8,2024,EVENT,March 2024 is when Meridian Health Systems ann...
9,2025,EVENT,"By the end of 2025, NovaCare Robotics emerged ..."


## Visualizing the graph

`entities.parquet` and `relationships.parquet` are, structurally, just a node table and an edge table — so
you can point *any* graph tool at them, not only GraphRAG's own search functions. Two options below: a quick
interactive view right here in the notebook, and exporting to a format a dedicated desktop tool can open.

### Option A — Interactive view inline (good for small/medium graphs)

[`pyvis`](https://pyvis.readthedocs.io/) turns a NetworkX graph into a zoomable, draggable HTML view — nodes
colored by community, sized by rank (roughly, how well-connected they are), with hover tooltips showing each
entity's type/description and each relationship's description.


In [5]:
# !pip install -q pyvis networkx


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Quick check of what columns are actually present in your indexed output —
# GraphRAG's schema has some optional columns that vary by version/config.
print("entities columns     :", list(entities.columns))
print("relationships columns:", list(relationships.columns))

entities columns     : ['id', 'human_readable_id', 'title', 'type', 'description', 'text_unit_ids', 'frequency', 'degree']
relationships columns: ['id', 'human_readable_id', 'source', 'target', 'description', 'weight', 'combined_degree', 'text_unit_ids']


In [11]:
import networkx as nx
from pyvis.network import Network

# Edge labels are relationship descriptions truncated to this length — full text still
# shows on hover. Set SHOW_EDGE_LABELS = False if a dense graph gets too cluttered to read.
SHOW_EDGE_LABELS = True
EDGE_LABEL_MAX_CHARS = 40

def _truncate(text, max_chars):
    text = text or ""
    return text if len(text) <= max_chars else text[: max_chars - 1] + "…"

G = nx.Graph()

for _, row in entities.iterrows():
    community_ids = row.get("community_ids")
    community = community_ids[0] if isinstance(community_ids, (list, tuple)) and len(community_ids) > 0 else -1
    rank = row.get("rank") or 1   # not every GraphRAG version/config populates this column
    G.add_node(
        row.get("title", "unknown"),
        title=f"{row.get('type', '')}: {row.get('description', '')}",   # pyvis hover tooltip
        group=community,                                                 # colors nodes by community
        size=10 + rank * 2,
    )

for _, row in relationships.iterrows():
    source, target = row.get("source"), row.get("target")
    if source in G.nodes and target in G.nodes:
        weight = row.get("weight") or 1.0
        description = row.get("description", "") or ""
        edge_kwargs = {"value": weight, "title": description}   # title = full text on hover
        if SHOW_EDGE_LABELS:
            edge_kwargs["label"] = _truncate(description, EDGE_LABEL_MAX_CHARS)  # visible on the edge itself
        G.add_edge(source, target, **edge_kwargs)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# net = Network(height="700px", width="100%", notebook=True, cdn_resources="in_line")
# net.from_nx(G)
# net.show("graph_visualization.html")


Graph: 10 nodes, 10 edges


### Export for a dedicated graph tool (better for large/complex graphs)

[Gephi](https://gephi.org/) (free, desktop) is built specifically for exploring and laying out large networks —
filtering by community, running layout algorithms, sizing/coloring by any column, etc. `networkx` can export the
exact same graph built above to GEXF, Gephi's native format


You can also use the free online tool to view the graph: https://lite.gephi.org/v1.0.2/


In [9]:
nx.write_gexf(G, "graph_export.gexf")
print("Wrote graph_export.gexf — open this file directly in Gephi (File > Open).")

Wrote graph_export.gexf — open this file directly in Gephi (File > Open).


## Global Search — holistic, corpus-wide questions

Global Search answers questions about the dataset *as a whole* by reasoning over the community reports
generated during indexing, rather than searching individual chunks. Good for "what are the big themes here?"
style questions that no single passage could answer alone.

`community_level` selects which level of the community hierarchy to summarize from — higher numbers mean
smaller, more granular communities; `2` is a reasonable default to start with.


In [12]:
response, context = asyncio.run(api.global_search(
    config=config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    community_level=2,
    dynamic_community_selection=False,
    response_type="Multiple Paragraphs",
    query="What are the main themes covered in this dataset?",
    verbose=False,
))

print(response)


## Main Themes in Healthcare Innovation and Robotic Surgery

The dataset presents several critical themes related to healthcare innovation, particularly emphasizing the role of robotic surgery. Below, we explore the prominent themes identified through the dataset.

### 1. **Impact of Robotic Surgery on Healthcare**

The dataset underscores the transformative impact of robotic surgery on healthcare practices. Notably, key figures like Wei Zhang and influential organizations such as NovaCare Robotics and Meridian Health Systems are highlighted for their contributions. Robotic assistance has been shown to enhance surgical precision, leading to improved patient outcomes. This evolution in surgical practices is crucial for understanding the future landscape of healthcare technology [Data: Reports (2, 1, 0)].

### 2. **Partnerships and Collaborations**

Strategic alliances within the healthcare technology sector are essential for fostering innovation. The collaborations between NovaCare Robo

## Local Search — specific, entity-centered questions

Local Search starts from the entities most relevant to your query, then pulls in their relationships and
supporting source text — better suited to questions about a specific person, organization, or concept in your
data. Replace the query below with something specific to your own documents.


In [4]:
response, context = asyncio.run(api.local_search(
    config=config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    text_units=text_units,
    relationships=relationships,
    covariates=None,
    community_level=2,
    response_type="Multiple Paragraphs",
    query="Tell me about Dr. Osei ",
    verbose=False,
))

print(response)


## Overview of Dr. Amara Osei

Dr. Amara Osei is the Chief Innovation Officer at Meridian Health Systems, a prominent hospital network located in Austin, Texas. She plays an essential role in advancing the integration of robotic surgery technologies within the healthcare system, specifically through her leadership in partnerships with companies like NovaCare Robotics. 

### Background and Expertise

Prior to her current position at Meridian Health Systems, Dr. Osei was involved in a similar robotics initiative at Cedar Ridge Medical. This background has provided her with significant expertise and a comprehensive understanding of the complexities involved in implementing robotic surgery, as well as the challenges faced when introducing new technologies in healthcare settings. Her experience uniquely positions her to navigate operational hurdles and align technological advances with regulatory standards and patient care priorities [Data: Entities (2); Reports (1, 0)].

### Key Contributi

## Bonus: Basic Search — a plain vector-search baseline

Basic Search skips the graph entirely and just does top-k similarity search over the raw text chunks —
useful as a quick sanity check or baseline to compare Global/Local Search against.


In [14]:
response, context = asyncio.run(api.basic_search(
    config=config,
    text_units=text_units,
    response_type="Multiple Paragraphs",
    query="Which part of USA would NovaCare Robotics become main supplier of its robots?",
    verbose=False,
))

print(response)


### NovaCare Robotics' Market Position

NovaCare Robotics has successfully established itself as the leading supplier of surgical robots specifically in the Southern United States. This significant milestone was reached by the end of 2025, following key partnerships and initiatives that enhanced its market presence and product reputation.

### Partnership with Meridian Health Systems

A pivotal moment for NovaCare Robotics was its partnership with Meridian Health Systems in March 2024, where it aimed to deploy surgical assistance robots across twelve facilities in Austin, Texas. This collaboration not only provided NovaCare with a broader platform to showcase its technology but also positioned it as a trusted name within the regional healthcare system [Data: Sources (0)].

### Impact of Community Engagement

The initial challenges faced by NovaCare, including concerns raised by the Texas Nurses Coalition regarding job displacement and patient safety, necessitated a proactive response f

## Bonus: DRIFT Search — the most thorough (and most expensive) method

DRIFT combines Local Search's entity focus with Global Search's community context, iteratively refining the
answer. It's the slowest and priciest of the four — reach for it when the other methods aren't giving you
enough depth on entity-specific questions.


In [15]:
response, context = asyncio.run(api.drift_search(
    config=config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    text_units=text_units,
    relationships=relationships,
    community_level=2,
    response_type="Multiple Paragraphs",
    query="Tell me about TEXAS NURSES COALITION",
    verbose=False,
))

print(response)


# Overview of the Texas Nurses Coalition

The Texas Nurses Coalition (TNC) is an influential organization dedicated to advocating for the interests and welfare of nurses across Texas. Established to unify the nursing profession, the Coalition focuses on various critical issues affecting nurses, including job security, patient safety, and the integration of technology in healthcare, particularly robotic surgery.

## Advocacy and Representation

The TNC plays a vital role in representing nurses' voices in discussions about healthcare policies and practices. Their advocacy efforts have been particularly pronounced in the context of the integration of robotic surgery systems, where they have raised concerns about potential job displacement for nursing staff and the implications for patient care. During public hearings, the Coalition has emphasized the need for a balanced approach to technology integration, ensuring that patient care remains a priority while also considering the stability o

## CLI equivalents

Everything above is also available as a one-liner from the terminal, for quick ad-hoc questions outside a
notebook:

```bash
graphrag query "What are the main themes?" --root ./graphrag_project --method global
graphrag query "Tell me about X" --root ./graphrag_project --method local
graphrag query "What are the main themes?" --root ./graphrag_project --method basic
graphrag query "Tell me about X" --root ./graphrag_project --method drift
```

## Wrap-up

- **Global** → broad, dataset-level questions.
- **Local** → specific entity/relationship questions.
- **Basic** → cheap vector-search baseline.
- **DRIFT** → most thorough entity-level answers, at the highest cost/latency.

Start with Global and Local for most use cases; reach for Basic as a sanity-check baseline and DRIFT when Local
Search isn't giving you enough depth.
